## Problem set 2, Parts 2 and 3: the consumption-savings problem, five ways (template)

Computational Macro (HWS 2026), University of Mannheim

*Team: name both team members here.*

One notebook, two models, five solvers. The solvers are written once, in sections 3, and take the
model as **arguments**: the utility function, its derivative and inverse derivative, the income
chain, the admissible consumption at every state, and a starting policy. Part 2 (section 4) passes
them the CARA household of Part 1 with persistent income, for which the exact policy of the
discretized problem is known, so that every method gets a true error. Part 3 (section 5) passes
them a CRRA household with log normal income and a borrowing limit, for which no exact policy exists and the Euler equation error is the only check.

**How to work with this notebook**

- Run the cells from top to bottom. After a restart of the kernel, start again at the top.
- Gaps are marked `...  # GAP k: ...`. The comment says what goes in and which row of the comparison
  table of Question 3.1 it belongs to. Cells without a gap run as they are. Read them anyway: the
  algorithm of every solver is stated above its loop, numbered as on the slides.
- Each part ends with a **Check**. Do not move on before it fits.
- Every reported result starts with the line that `report` prints from `par` and `mpar`.

| Section | Content | Gaps | Question |
| --- | --- | --- | --- |
| 1 | Structures: parameters, grids, the report line | 3 | |
| 2 | Tools: the chain, interpolation, golden section search, expectations, bisection, the error measures | | |
| 3 | The five solvers: VFI on the grid, VFI off the grid, Howard, time iteration, EGM | | |
| 4 | Part 2: the CARA household with persistent income, its exact policy, the five methods | | 2.1, 2.2 |
| 5 | Part 3: the CRRA household with log normal income and a borrowing limit | 8 | 3.2 and 3.3 |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\mathcal{A} = \{a_1, \ldots, a_{N_a}\}$, $\mathcal{Z} = \{z_1, \ldots, z_{N_z}\}$ | asset and income grids, income in levels in both models | `gri.a`, `gri.z` |
| $\boldsymbol{\Pi}$, $\pi_{mn} = \Pr(z' = z_n \mid z = z_m)$ | transition matrix, rows sum to one | `gri.Pi` |
| $x_{im} = z_m + R a_i$ | cash on hand, an $N_a \times N_z$ array | `gri.x` |
| $c^{\min}_{im}, c^{\max}_{im}$ | admissible consumption at state $(a_i, z_m)$, from the grid and the model | `cmin`, `cmax` from `bounds(par, mpar, gri)` |
| $\mathbf{V}$, $V_{im} = v(a_i, z_m)$ | value function on the grid | `V` |
| $a'_{im}$, $c_{im}$ | savings and consumption policy | `aprime`, `c` |
| $\mathbf{EV} = \beta \mathbf{V} \boldsymbol{\Pi}^\top$ | expected continuation value, $EV_{lm} = \beta \sum_n V_{ln} \pi_{mn}$ | `par.beta * V @ gri.Pi.T` |
| $\hat v_n(a')$, $\sum_n \pi_{mn} \hat v_n(a'_{im})$ | interpolant of column $n$ of $\mathbf{V}$, and the expectation at any choice | `expected_value(V, gri, mpar, aprime)` |
| $\mathbf{EMU} = u_c(\mathbf{c}) \boldsymbol{\Pi}^\top$ | expected marginal utility tomorrow at every $(a'_l, z_m)$ | `expected_mu(C, par, gri, uprime)` |
| $f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c)$ | the time iteration residual | `resid` |
| $\tilde c_{lm}, \tilde a_{lm}$ | the endogenous grid of one EGM step | `ctilde`, `atilde` |
| $c^{\text{exact}}_{im}$, $e_W$, $e_G$ | exact policy of the discretized problem, largest error on the window and on the grid | `c_exact(par, gri)`, `errors(...)` |
| $e_{im}$ | Euler equation error in units of consumption | `euler_error(...)`, `euler_stats(...)` |

### 0. Packages

The same as in week 3 and on Thursday. The spline comes from `scipy`, everything else is written here.

In [ ]:
# Installs the packages of this notebook into the environment of the running kernel.
%pip install "numpy==2.2.*" "matplotlib==3.10.*" "scipy==1.15.*"

In [ ]:
import time
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

### 1. Structures

Three containers, as in the problem sets before: `par` holds the economics of **one model** and is
therefore defined per model (`CARAParameters` here, `CRRAParameters` in Part 3), `mpar` holds the
numerics and is shared, `gri` holds the grids and cash on hand. The solvers never look inside
`par` except for `beta` and `R`, everything else about the model reaches them through the
functions you pass as arguments.

**The stopping rules (gaps 1 to 3).** The set states them in words: the two value function
iterations stop when the value moves by less than $\varepsilon$, the three policy methods (Howard,
time iteration, EGM) stop when the policy moves by less than $\varepsilon_{\text{pol}}$, and every
loop has an iteration cap of $5000$ as a safety net. Translate them into the three fields.

In [ ]:
@dataclass(frozen=True)
class CARAParameters:
    # The household of Part 1: CARA utility, income z = zbar + e with e persistent and normal
    gamma: float = 2.0        # absolute risk aversion
    r: float = 0.04           # net interest rate, R = 1 + r
    beta: float = 1 / 1.04    # discount factor: beta R = 1, no impatience, saving is precautionary only
    zbar: float = 1.0         # mean income
    sigma: float = 0.1        # unconditional standard deviation of e
    rho: float = 0.9          # persistence of e

    @property
    def R(self):
        return 1 + self.r


@dataclass(frozen=True)
class NumericalParameters:
    na: int = 400             # asset grid points N_a
    nz: int = 7               # income grid points N_z
    amin: float = -15.0       # lower grid bound a_1
    amax: float = 25.0        # upper grid bound a_{N_a}
    spacing: str = "linear"   # "linear": equidistant, "quadratic": denser near a_1
    crit: float = ...  # GAP 1: tolerance on the value function, epsilon, for VFI on and off the grid
    crit_pol: float = ...  # GAP 2: tolerance on the policy, epsilon_pol, for Howard, time iteration, and EGM
    maxiter: int = ...  # GAP 3: the iteration cap, the safety net of every loop
    n_howard: int = 50        # evaluation sweeps per maximization in Howard's algorithm
    nbisect: int = 60         # bisection steps per time iteration step
    interp: str = "spline"    # how V is evaluated between grid points: "linear" or "spline"
    awin_lo: float = 0.0      # errors are reported on the window awin_lo <= a <= awin_hi
    awin_hi: float = 8.0


@dataclass(frozen=True)
class Grids:
    a: np.ndarray             # asset grid, N_a
    z: np.ndarray             # income grid in levels, N_z (both models)
    Pi: np.ndarray            # transition matrix, N_z x N_z
    x: np.ndarray             # cash on hand x_im = z_m + R a_i, N_a x N_z (both models)


def report(name, res, par, mpar):
    # The header line that precedes every reported result, printed from par and mpar, and the diagnostics
    print(f"{name} with {par}, N_a = {mpar.na}, N_z = {mpar.nz}, grid = [{mpar.amin}, {mpar.amax}] ({mpar.spacing}), "
          f"crit = {mpar.crit:g}, crit_pol = {mpar.crit_pol:g}")
    print(f"  iterations: {res['iters']}, converged: {res['converged']}, final distance: {res['dist'][-1]:.1e}, "
          f"time: {res['time']:.2f} s, {1000 * res['time'] / res['iters']:.2f} ms per iteration")
    print(f"  policy at a_1: {100 * res['at_lower']:.1f} % of states, at a_N: {100 * res['at_upper']:.1f} % of states")

### 2. Tools

Nothing to fill in. Each tool comes from one slide or one earlier notebook.

| Tool | From | Used by |
| --- | --- | --- |
| `rouwenhorst`, `stationary` | session 5, slides 5 and 6 | the income functions of both models |
| `make_grids` | week 4 | everything: builds $\mathcal{A}$, calls the model's `income` for $\mathcal{Z}$ and $\boldsymbol{\Pi}$, and forms $x_{im}$ |
| `make_interpolant` | week 3 | VFI off the grid, Howard |
| `golden_max_vec` | week 3, all states at once | VFI off the grid, Howard |
| `expected_value` | session 4, $\sum_n \pi_{mn} \hat v_n(a'_{im})$ | VFI off the grid, Howard |
| `expected_mu` | session 5, $\mathbf{EMU} = u_c(\mathbf{c}) \boldsymbol{\Pi}^\top$ | time iteration, EGM |
| `bisect_vec` | session 5, backup slide | time iteration |
| `errors` | slide 22 | Part 2, the error against the exact policy |
| `euler_error`, `euler_stats` | problem set 1 | both parts |

In [ ]:
def rouwenhorst(n, rho, sigma):
    # Markov chain on n points for y' = rho y + eps, built so that sd(y) = sigma and autocorrelation rho (slides 5 and 6)
    p = (1 + rho) / 2
    Pi = np.array([[p, 1 - p], [1 - p, p]])
    for m in range(3, n + 1):                                # grow the chain one state at a time
        P = np.zeros((m, m))
        P[:-1, :-1] += p * Pi
        P[:-1, 1:] += (1 - p) * Pi
        P[1:, :-1] += (1 - p) * Pi
        P[1:, 1:] += p * Pi
        P[1:-1, :] /= 2                                      # the interior rows were counted twice
        Pi = P
    psi = sigma * np.sqrt(n - 1)                             # the grid is equidistant on [-psi, psi]
    return np.linspace(-psi, psi, n), Pi


def stationary(Pi):
    # the stationary distribution: the eigenvector of Pi^T to the eigenvalue one, normalized to sum to one
    w, v = np.linalg.eig(Pi.T)
    lam = np.real(v[:, np.argmin(np.abs(w - 1))])
    return lam / lam.sum()


def make_grids(par, mpar, income):
    # The asset grid, the income chain from the model's income function, and cash on hand
    z, Pi = income(par, mpar)                                # income in levels and the transition matrix
    u = np.linspace(0.0, 1.0, mpar.na)
    if mpar.spacing == "quadratic":
        u = u ** 2                                           # denser near a_1
    a = mpar.amin + (mpar.amax - mpar.amin) * u
    x = z[None, :] + par.R * a[:, None]                      # x_im = z_m + R a_i, state i in the rows, income m in the columns
    return Grids(a=a, z=z, Pi=Pi, x=x)

In [ ]:
def make_interpolant(xgrid, fgrid, method):
    # week 3: straight lines or the natural cubic spline through the points (xgrid, fgrid)
    if method == "linear":
        return lambda x: np.interp(x, xgrid, fgrid)
    return CubicSpline(xgrid, fgrid, bc_type="natural")


def golden_max_vec(f, a, b, tol=1e-10):
    # golden section search on every entry of the arrays a < b at once, same algorithm as in week 3
    phi = (np.sqrt(5) - 1) / 2
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    c, d = b - phi * (b - a), a + phi * (b - a)
    fc, fd = f(c), f(d)
    nsteps = int(np.ceil(np.log(tol / np.max(b - a)) / np.log(phi)))   # steps until every bracket is narrower than tol
    for _ in range(nsteps):
        left = fc > fd                                       # True where the maximum is in [a, d], False where it is in [c, b]
        a, b = np.where(left, a, c), np.where(left, d, b)    # the new bracket
        c_new, d_new = b - phi * (b - a), a + phi * (b - a)  # the new interior points
        fnew = f(np.where(left, c_new, d_new))               # one new evaluation per state: c where left, d where right
        fc, fd = np.where(left, fnew, fd), np.where(left, fc, fnew)   # the other interior point is reused
        c, d = c_new, d_new
    x = (a + b) / 2
    return x, f(x)


def expected_value(V, gri, mpar, aprime):
    # sum_n pi_mn vhat_n(a'_im): one interpolant per column of V, weighted with row m of Pi (session 4, off the grid)
    EV = np.zeros_like(aprime)
    for n in range(gri.Pi.shape[0]):
        vhat_n = make_interpolant(gri.a, V[:, n], mpar.interp)(aprime)   # vhat_n at every a'_im, an N_a x N_z array
        EV += gri.Pi[:, n][None, :] * vhat_n                 # weight pi_mn: column n of Pi, one weight per income state m today
    return EV


def expected_mu(C, par, gri, uprime):
    # EMU[l, m] = sum_n pi_mn u_c(C[l, n]): expected marginal utility tomorrow for a household that saves a'_l
    # and has income z_m today, one matrix product (slide 12)
    return uprime(C, par) @ gri.Pi.T


def bisect_vec(f, lo, hi, steps):
    # bisection on every grid point at once (backup slide): f is decreasing, the root lies in [lo, hi] point by point
    for _ in range(steps):
        mid = (lo + hi) / 2
        pos = f(mid) > 0                                     # True where f(mid) > 0: the root is in [mid, hi]
        lo = np.where(pos, mid, lo)
        hi = np.where(pos, hi, mid)
    return (lo + hi) / 2

In [ ]:
def window(mpar, gri):
    # the rows of the grid on which errors are reported
    return (gri.a >= mpar.awin_lo) & (gri.a <= mpar.awin_hi)


def errors(c, cex, mpar, gri):
    # absolute consumption error against the exact policy cex: largest on the window (e_W), on the whole grid (e_G),
    # and the profile along the asset grid (largest over income states at each a_i)
    err = np.abs(c - cex)
    return dict(window=err[window(mpar, gri)].max(), all=err.max(), profile=err.max(axis=1))


def euler_error(c, par, mpar, gri, uprime, uprime_inv):
    # Euler equation error at every grid point, in units of consumption, as in problem set 1:
    #   e_im = 1 - u_c^{-1}( beta R sum_n pi_mn u_c(c_n(a'_im)) ) / c_im,
    # with c_n(.) the policy at income z_n interpolated linearly at the choice a'_im. NaN where a grid bound
    # binds, since there the Euler equation holds as an inequality.
    aprime = gri.x - c                                       # the choice at every state
    mu_next = np.zeros_like(c)                               # sum_n pi_mn u_c(c_n(a'_im)), filled one income state n at a time
    for n in range(mpar.nz):
        c_n = np.interp(aprime, gri.a, c[:, n])              # the policy of state n at every choice a'_im
        mu_next += gri.Pi[:, n][None, :] * uprime(c_n, par)  # weight pi_mn, one per income state m today
    e = 1 - uprime_inv(par.beta * par.R * mu_next, par) / c
    bound = (aprime <= gri.a[0] + 1e-8) | (aprime >= gri.a[-1] - 1e-8)
    return np.where(bound, np.nan, e)


def euler_stats(e, mpar, gri):
    # largest |e| and the mean of log10 |e| on the window, bound states (NaN) skipped
    ew = np.abs(e[window(mpar, gri)])
    ew = ew[~np.isnan(ew)]
    return ew.max(), np.mean(np.log10(np.maximum(ew, 1e-16)))

### 3. The five solvers

Nothing to fill in. Each solver takes `par`, `mpar`, `gri` and the model functions it needs, and
returns the same dictionary: `V` (`None` for the Euler equation methods), `aprime`, `c`, `iters`,
`time`, `dist` (the distances between iterates), `converged`, and the share of states at which the
policy sits at $a_1$ (`at_lower`) or at $a_{N_a}$ (`at_upper`). The argument list of each solver
is the list of what the method needs from the model.

**Method 1: value function iteration on the grid** (session 4). The consumption array
$C_{iml} = x_{im} - a_l$, the reward array $U_{iml} = u(C_{iml})$ where $c^{\min}_{im} \le C_{iml} \le c^{\max}_{im}$
and $-\infty$ elsewhere, both built once. Then

1. $\mathbf{EV} = \beta \mathbf{V} \boldsymbol{\Pi}^\top$
2. $M_{iml} = U_{iml} + EV_{lm}$
3. $V^{\text{new}}_{im} = \max_l M_{iml}$, $\ G_{im} = \arg\max_l M_{iml}$
4. stop if $\lVert \mathbf{V}^{\text{new}} - \mathbf{V} \rVert_\infty < \varepsilon$

The mask on $U$ is the only place where the admissible consumption enters. With CARA it never
bites, with CRRA it removes $c \le 0$.

In [ ]:
def vfi_on_grid(par, mpar, gri, util, bounds):
    t0 = time.perf_counter()
    cmin, cmax = bounds(par, mpar, gri)                      # admissible consumption at every state
    C = gri.x[:, :, None] - gri.a[None, None, :]             # C_iml = x_im - a_l: state i, income m, choice l
    feasible = (C >= cmin[:, :, None]) & (C <= cmax[:, :, None])
    U = np.where(feasible, util(np.where(feasible, C, 1.0), par), -np.inf)   # u where feasible, -inf elsewhere
    V = np.zeros((mpar.na, mpar.nz))
    dist, it = [np.inf], 0
    while dist[-1] > mpar.crit and it < mpar.maxiter:
        EV = par.beta * V @ gri.Pi.T                         # EV_lm = beta sum_n V_ln pi_mn, an N_a x N_z array
        M = U + EV.T[None, :, :]                             # M_iml = U_iml + EV_lm: EV transposed so that its indices are (m, l)
        Vnew, G = M.max(axis=2), M.argmax(axis=2)            # the maximum and the maximizing index along the choice axis
        dist.append(np.max(np.abs(Vnew - V)))
        V = Vnew
        it += 1
    aprime = gri.a[G]                                        # the policy in levels
    c = gri.x - aprime
    return dict(V=V, aprime=aprime, c=c, iters=it, time=time.perf_counter() - t0, dist=dist[1:],
                converged=dist[-1] <= mpar.crit, at_lower=np.mean(G == 0), at_upper=np.mean(G == mpar.na - 1))

**Method 2: value function iteration off the grid** (week 3 and session 4). For every state the
right hand side is a function of the continuous choice $a'$,

$$
a' \mapsto u(x_{im} - a') + \beta \sum_n \pi_{mn} \hat v_n(a'),
$$

maximized by golden section search on the bracket $[x_{im} - c^{\max}_{im},\ x_{im} - c^{\min}_{im}]$,
the admissible consumption translated into tomorrow's assets. With `golden_max_vec` all states are
searched at once. `bellman_max` is the building block of Howard's algorithm as well.

1. one maximization at every state: $\mathbf{V}^{\text{new}}$ and the policy $\mathbf{a}'$
2. stop if $\lVert \mathbf{V}^{\text{new}} - \mathbf{V} \rVert_\infty < \varepsilon$

In [ ]:
def bellman_max(V, par, mpar, gri, util, bounds):
    # one maximization step off the grid: the new value and the policy a', for all states at once
    cmin, cmax = bounds(par, mpar, gri)
    lo, hi = gri.x - cmax, gri.x - cmin                      # the bracket on a': consume the most, save the most
    rhs = lambda ap: util(gri.x - ap, par) + par.beta * expected_value(V, gri, mpar, ap)   # the right hand side of the Bellman equation
    aprime, Vnew = golden_max_vec(rhs, lo, hi)
    return Vnew, aprime


def vfi_off_grid(par, mpar, gri, util, bounds):
    t0 = time.perf_counter()
    V = np.zeros((mpar.na, mpar.nz))
    dist, it = [np.inf], 0
    while dist[-1] > mpar.crit and it < mpar.maxiter:
        Vnew, aprime = bellman_max(V, par, mpar, gri, util, bounds)
        dist.append(np.max(np.abs(Vnew - V)))
        V = Vnew
        it += 1
    c = gri.x - aprime
    return dict(V=V, aprime=aprime, c=c, iters=it, time=time.perf_counter() - t0, dist=dist[1:],
                converged=dist[-1] <= mpar.crit, at_lower=np.mean(aprime <= gri.a[0] + 1e-8), at_upper=np.mean(aprime >= gri.a[-1] - 1e-8))

**Method 3: Howard's improvement algorithm** (week 3, part 5, and the session 5 backup slide).
A Bellman step uses the new policy for one period and then continues with the old value. Howard's
idea: after the maximization, keep the policy fixed and update the value $n_H$ times without
maximizing,

$$
V_{im} \leftarrow u(x_{im} - a'_{im}) + \beta \sum_n \pi_{mn} \hat v_n(a'_{im}),
$$

which is cheap, because the expensive part of a Bellman step is the search. This is the modified
policy iteration of the backup slide: the $n_H$ sweeps replace the linear solve
$(\mathbf{I} - \beta \mathbf{Q}_{\mathbf{g}}) \mathbf{v} = \mathbf{u}_{\mathbf{g}}$, which here would have
$(N_a N_z)^2$ entries.

1. maximize as in method 2: policy $\mathbf{a}'$ and the reward $\mathbf{u} = u(\mathbf{x} - \mathbf{a}')$
2. $n_H$ sweeps $\mathbf{V} \leftarrow \mathbf{u} + \beta\, \sum_n \pi_{mn} \hat v_n(a'_{im})$ with the policy held fixed
3. stop if $\lVert \mathbf{a}'^{\text{new}} - \mathbf{a}' \rVert_\infty < \varepsilon_{\text{pol}}$

In [ ]:
def howard(par, mpar, gri, util, bounds):
    t0 = time.perf_counter()
    V = np.zeros((mpar.na, mpar.nz))
    aprime_old = np.full((mpar.na, mpar.nz), np.nan)
    dist, it = [np.inf], 0
    while dist[-1] > mpar.crit_pol and it < mpar.maxiter:
        V, aprime = bellman_max(V, par, mpar, gri, util, bounds)   # improvement: one maximization, as in method 2
        u = util(gri.x - aprime, par)                        # the reward under the policy, fixed during the sweeps
        for _ in range(mpar.n_howard):                       # evaluation: n_H sweeps with the policy held fixed
            V = u + par.beta * expected_value(V, gri, mpar, aprime)
        dist.append(np.max(np.abs(aprime - aprime_old)) if it > 0 else np.inf)   # how much did the policy move?
        aprime_old = aprime
        it += 1
    c = gri.x - aprime
    return dict(V=V, aprime=aprime, c=c, iters=it, time=time.perf_counter() - t0, dist=dist[1:],
                converged=dist[-1] <= mpar.crit_pol, at_lower=np.mean(aprime <= gri.a[0] + 1e-8), at_upper=np.mean(aprime >= gri.a[-1] - 1e-8))

**Method 4: time iteration** (session 5, slides 11 to 14, Thursday notebook part 2). Policy
function iteration on the Euler equation: guess the policy, and at every state find the $c$ that
solves the Euler equation when tomorrow's household follows the guess.

1. guess $\mathbf{c}_0$, the model's `guess`, clipped to the admissible range
2. $\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$
3. the residual $f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c)$, with column $m$ of $\mathbf{EMU}$ interpolated at $a' = x_{im} - c$
4. the bracket is the admissible consumption $[c^{\min}_{im}, c^{\max}_{im}]$: if $f_{im}(c^{\max}) \ge 0$ the household wants to borrow more and $c = c^{\max}$, if $f_{im}(c^{\min}) \le 0$ it wants to save more and $c = c^{\min}$
5. otherwise the root by bisection, all states at once
6. stop if $\lVert \mathbf{c}_{n+1} - \mathbf{c}_n \rVert_\infty < \varepsilon_{\text{pol}}$

In [ ]:
def resid(c, EMU, par, gri, uprime):
    # the Euler equation residual f_im(c) = u_c(c) - beta R EMUhat_m(x_im - c), for an N_a x N_z array c
    aprime = gri.x - c                                       # tomorrow's assets implied by c, in general off the grid
    EMUhat = np.empty_like(c)
    for m in range(len(gri.z)):                              # one income state today at a time
        EMUhat[:, m] = np.interp(aprime[:, m], gri.a, EMU[:, m])
    return uprime(c, par) - par.beta * par.R * EMUhat


def time_iteration_step(C, par, mpar, gri, uprime, bounds):
    # one step of time iteration: the new policy from the guess C
    EMU = expected_mu(C, par, gri, uprime)                   # step 2
    f = lambda c: resid(c, EMU, par, gri, uprime)            # step 3
    cmin, cmax = bounds(par, mpar, gri)                      # step 4: the bracket, and where a limit binds
    borrow = f(cmax) >= 0                                    # even cmax is too little: the borrowing limit binds
    save = f(cmin) <= 0                                      # even cmin is too much: the saving limit binds
    c_root = bisect_vec(f, cmin, cmax, mpar.nbisect)         # step 5: the root at every state
    Cnew = np.where(save, cmin, c_root)
    return np.where(borrow, cmax, Cnew)


def time_iteration(par, mpar, gri, uprime, bounds, guess):
    t0 = time.perf_counter()
    cmin, cmax = bounds(par, mpar, gri)
    C = np.clip(guess(par, gri), cmin, cmax)                 # step 1: the model's guess, inside the admissible range
    dist, it = [np.inf], 0
    while dist[-1] > mpar.crit_pol and it < mpar.maxiter:
        Cnew = time_iteration_step(C, par, mpar, gri, uprime, bounds)
        dist.append(np.max(np.abs(Cnew - C)))
        C = Cnew
        it += 1
    aprime = gri.x - C
    return dict(V=None, aprime=aprime, c=C, iters=it, time=time.perf_counter() - t0, dist=dist[1:],
                converged=dist[-1] <= mpar.crit_pol, at_lower=np.mean(aprime <= gri.a[0] + 1e-8), at_upper=np.mean(aprime >= gri.a[-1] - 1e-8))

**Method 5: the endogenous grid method** (session 5, slides 17 to 20, Thursday notebook part 3).
Fix tomorrow's assets on the grid, $a' = a'_l$. Then the Euler equation gives consumption today
without a root, and the budget constraint gives the asset level today at which the household
chooses $a'_l$.

1. guess $\mathbf{c}_0$
2. $\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$ and $\tilde c_{lm} = u_c^{-1}(\beta R \, EMU_{lm})$
3. the endogenous grid $\tilde a_{lm} = (\tilde c_{lm} + a'_l - z_m) / R$
4. for each $m$, interpolate $(\tilde a_{\cdot m}, \tilde c_{\cdot m})$ at $\mathcal{A}$ with straight lines
5. the limits: below $\tilde a_{1m}$, $c = x_{im} - a_1$. Above $\tilde a_{N_a m}$, $c = x_{im} - a_{N_a}$
6. stop if $\lVert \mathbf{c}_{n+1} - \mathbf{c}_n \rVert_\infty < \varepsilon_{\text{pol}}$

No `bounds` argument: the inverted marginal utility returns admissible consumption by itself, and
at the limits consumption is $x - a_1$ or $x - a_{N_a}$, the ends of the bracket. The limits come
for free (slide 19).

In [ ]:
def egm_step(C, par, mpar, gri, uprime, uprime_inv):
    # one step of the endogenous grid method: the new policy on the grid, and the endogenous grid of this step
    EMU = expected_mu(C, par, gri, uprime)                   # step 2
    ctilde = uprime_inv(par.beta * par.R * EMU, par)         # consumption today from the Euler equation, no root
    atilde = (ctilde + gri.a[:, None] - gri.z[None, :]) / par.R   # step 3: where did the household start?
    Cnew = np.empty_like(C)
    for m in range(len(gri.z)):                              # step 4: back to the grid, one income state at a time
        Cnew[:, m] = np.interp(gri.a, atilde[:, m], ctilde[:, m])
        below, above = gri.a < atilde[0, m], gri.a > atilde[-1, m]   # step 5: where the endogenous grid ends
        Cnew[below, m] = gri.x[below, m] - gri.a[0]          # the borrowing limit binds
        Cnew[above, m] = gri.x[above, m] - gri.a[-1]         # the saving limit binds
    return Cnew, ctilde, atilde


def egm(par, mpar, gri, uprime, uprime_inv, guess):
    t0 = time.perf_counter()
    C = np.clip(guess(par, gri), gri.x - gri.a[-1], gri.x - gri.a[0])   # step 1: the guess, inside the grid
    dist, it = [np.inf], 0
    while dist[-1] > mpar.crit_pol and it < mpar.maxiter:
        Cnew, _, _ = egm_step(C, par, mpar, gri, uprime, uprime_inv)
        dist.append(np.max(np.abs(Cnew - C)))
        C = Cnew
        it += 1
    aprime = gri.x - C
    return dict(V=None, aprime=aprime, c=C, iters=it, time=time.perf_counter() - t0, dist=dist[1:],
                converged=dist[-1] <= mpar.crit_pol, at_lower=np.mean(aprime <= gri.a[0] + 1e-8), at_upper=np.mean(aprime >= gri.a[-1] - 1e-8))

### 4. Part 2: the CARA household with persistent income

*Questions 2.1 and 2.2.* The household of Part 1: $u(c) = -e^{-\gamma c}/\gamma$, defined for every
real $c$, income $z = \bar z + e$ with $e' = \rho e + \sigma_\varepsilon \varepsilon'$, no borrowing
limit. On the computer $e$ lives on a Rouwenhorst chain with $N_z = 7$ points, unconditional
standard deviation $\sigma$ and persistence $\rho$, and the asset grid imposes $a_1 \le a' \le a_{N_a}$.

**The model as functions.** Everything the solvers need to know about this household, nothing
to fill in. Note `bounds_cara`: the admissible consumption at every state follows from
$c + a' = x$ and $a_1 \le a' \le a_{N_a}$, nothing else restricts $c$ in this model.

In [ ]:
def util_cara(c, par):
    return -np.exp(-par.gamma * c) / par.gamma


def uprime_cara(c, par):
    return np.exp(-par.gamma * c)


def uprime_inv_cara(m, par):
    return -np.log(m) / par.gamma


def income_level(par, mpar):
    # income in levels: z = zbar + e, with e on a Rouwenhorst chain with sd sigma and persistence rho
    e, Pi = rouwenhorst(mpar.nz, par.rho, par.sigma)
    return par.zbar + e, Pi


def bounds_cara(par, mpar, gri):
    # admissible consumption at every state (a_i, z_m), two N_a x N_z arrays. Only the grid restricts a'
    cmin = gri.x - gri.a[-1]                                 # the household saves the most it can, a' = a_N
    cmax = gri.x - gri.a[0]                                  # the household borrows the most it can, a' = a_1
    return cmin, cmax


def guess_cara(par, gri):
    # starting policy for the Euler equation methods: the exact policy at sigma = 0 (certainty equivalence)
    mu = par.r / par.R
    nu = par.rho * mu / (par.R - par.rho)
    return mu * gri.x + nu * (gri.z - par.zbar)[None, :] + par.zbar / par.R

**The exact policy of the discretized problem (Question 2.1 a).** With $e$ on the chain, the
conditional expectation in the Euler equation is a weighted sum, and for a Rouwenhorst chain its
logarithm is affine in today's state,

$$
\ln \sum_n \pi_{mn} e^{s e_n} = K(s) + S(s)\, e_m \qquad \text{for every } m,
$$

where for the normal distribution $K(s) = s^2 \sigma_\varepsilon^2 / 2$ and $S(s) = s \rho$. The
coefficient matching of Question 1.2 (b) then gives $\mu = r/R$ unchanged, $\nu$ as the solution of

$$
\gamma R \, \nu = -S\bigl(-\gamma (\mu + \nu)\bigr),
$$

and $f_{N_z} = \bigl[\gamma \mu \bar z - \ln(\beta R) - K(-\gamma(\mu + \nu))\bigr] / (\gamma r)$.
`exact_coefficients` checks the affinity, solves the $\nu$ equation by bisection (the root finder
of the slides), and `c_exact` returns $c^{\text{exact}}_{im} = \mu x_{im} + \nu e_m + f_{N_z}$.
Nothing to fill in.

In [ ]:
def exact_coefficients(par, gri):
    # mu, nu, f of the exact policy of the discretized problem, and the largest deviation of ln M_m(s) from a line in e_m
    R, r, g = par.R, par.r, par.gamma
    mu = r / R
    e = gri.z - par.zbar
    lnM = lambda s: np.log(gri.Pi @ np.exp(s * e))           # ln sum_n pi_mn exp(s e_n), one number per state m
    S = lambda s: (lnM(s)[-1] - lnM(s)[0]) / (e[-1] - e[0])  # slope in e_m, from the two end states
    K = lambda s: lnM(s)[0] - S(s) * e[0]                    # intercept
    lo, hi = 0.0, 1.0                                        # nu solves g(nu) = gamma R nu + S(-gamma (mu + nu)) = 0, g increasing
    for _ in range(60):
        mid = (lo + hi) / 2
        if g * R * mid + S(-g * (mu + mid)) > 0:
            hi = mid
        else:
            lo = mid
    nu = (lo + hi) / 2
    s = -g * (mu + nu)
    f = (g * mu * par.zbar - np.log(par.beta * R) - K(s)) / (g * r)
    dev = np.max(np.abs(lnM(s) - K(s) - S(s) * e))           # the affinity check: zero up to rounding for Rouwenhorst
    return mu, nu, f, dev


def c_exact(par, gri):
    # the exact policy of the discretized problem on the grid
    mu, nu, f, _ = exact_coefficients(par, gri)
    return mu * gri.x + nu * (gri.z - par.zbar)[None, :] + f


def closed_form_gauss(par):
    # mu, nu, f of Part 1 for normally distributed income
    R, r = par.R, par.r
    mu = r / R
    nu = par.rho * mu / (R - par.rho)
    sig_eps2 = par.sigma ** 2 * (1 - par.rho ** 2)
    f = par.zbar / R - np.log(par.beta * R) / (par.gamma * r) - par.gamma * (mu + nu) ** 2 * sig_eps2 / (2 * r)
    return mu, nu, f

In [ ]:
par, mpar = CARAParameters(), NumericalParameters()
gri = make_grids(par, mpar, income_level)
lam = stationary(gri.Pi)
print(f"income grid z: {np.round(gri.z, 3)}")
print(f"stationary distribution: {np.round(lam, 4)}, mean {lam @ gri.z:.4f}, sd {np.sqrt(lam @ (gri.z - par.zbar) ** 2):.4f}")
print(f"row of Pi at the middle state: {np.round(gri.Pi[mpar.nz // 2], 4)}")
mu, nu, f, dev = exact_coefficients(par, gri)
mu_g, nu_g, f_g = closed_form_gauss(par)
print(f"affinity of ln M_m(s) in e_m: largest deviation from a line {dev:.1e}")
print(f"exact policy of the discretized problem: mu = {mu:.6f}, nu = {nu:.6f}, f = {f:.8f}")
print(f"Gaussian closed form of Part 1:          mu = {mu_g:.6f}, nu = {nu_g:.6f}, f = {f_g:.8f}")
print(f"difference in consumption at the extreme income states: {abs(nu - nu_g) * abs(gri.z[-1] - par.zbar) + abs(f - f_g):.1e}")
print(f"window: {window(mpar, gri).sum()} of {mpar.na} grid points, a in [{mpar.awin_lo}, {mpar.awin_hi}]")

**The five methods (Question 2.2 a).** Each solver is called with the CARA functions, `report`
prints the header line and the diagnostics, and the errors are computed against the exact policy
and with the Euler equation. The iteration counts in the table below are those of the correct
solution, the other columns are yours to fill in.

| method | iterations | time (s) | ms per iteration | $e_W$ | $e_G$ | Euler error, max | Euler error, mean $\log_{10}$ |
| --- | --- | --- | --- | --- | --- | --- | --- |
| VFI on the grid | 314 | | | | | | |
| VFI off the grid | 314 | | | | | | |
| Howard | 15 | | | | | | |
| Time iteration | 136 | | | | | | |
| EGM | 136 | | | | | | |

In [ ]:
cex = c_exact(par, gri)
results = {}
results["VFI on the grid"] = vfi_on_grid(par, mpar, gri, util_cara, bounds_cara)
results["VFI off the grid"] = vfi_off_grid(par, mpar, gri, util_cara, bounds_cara)
results["Howard"] = howard(par, mpar, gri, util_cara, bounds_cara)
results["Time iteration"] = time_iteration(par, mpar, gri, uprime_cara, bounds_cara, guess_cara)
results["EGM"] = egm(par, mpar, gri, uprime_cara, uprime_inv_cara, guess_cara)
for name, res in results.items():
    report(name, res, par, mpar)
    res.update(errors(res["c"], cex, mpar, gri))
    res["euler"] = euler_error(res["c"], par, mpar, gri, uprime_cara, uprime_inv_cara)
    res["ee_max"], res["ee_mean"] = euler_stats(res["euler"], mpar, gri)
    print(f"  largest consumption error: window {res['window']:.1e}, whole grid {res['all']:.1e}; "
          f"Euler error on the window: max {res['ee_max']:.1e}, mean log10 {res['ee_mean']:.1f}")

In [ ]:
print(f"{'method':18s} {'iterations':>10s} {'time (s)':>9s} {'ms/iter':>8s} {'e_W':>9s} {'e_G':>9s} {'EE max':>9s} {'EE mean log10':>14s}")
for name, res in results.items():
    print(f"{name:18s} {res['iters']:10d} {res['time']:9.2f} {1000 * res['time'] / res['iters']:8.2f} "
          f"{res['window']:9.1e} {res['all']:9.1e} {res['ee_max']:9.1e} {res['ee_mean']:14.1f}")

The consumption policy at the highest income state, on the whole grid and zoomed in, and the two
error measures along the asset grid on a log scale (the largest over income states at each $a_i$).

In [ ]:
top = mpar.nz - 1
zoom = (gri.a >= 3.0) & (gri.a <= 4.0)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, res in results.items():
    ax[0].plot(gri.a, res["c"][:, top], label=name)
    ax[1].plot(gri.a[zoom], res["c"][zoom, top], "o-" if name == "VFI on the grid" else "-", markersize=3, label=name)
ax[0].plot(gri.a, cex[:, top], "k--", label="exact policy")
ax[1].plot(gri.a[zoom], cex[zoom, top], "k--", label="exact policy")
ax[0].axvspan(mpar.awin_lo, mpar.awin_hi, color="gray", alpha=0.1)
ax[0].set(title=f"Consumption at z = {gri.z[top]:.2f}, whole grid (shaded: the window)", xlabel="assets a", ylabel="consumption c")
ax[1].set(title="Zoom, a in [3, 4]", xlabel="assets a")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, res in results.items():
    ax[0].semilogy(gri.a, np.maximum(res["profile"], 1e-16), label=name)
    ee = np.nanmax(np.abs(res["euler"]), axis=1)
    ax[1].semilogy(gri.a, np.maximum(ee, 1e-16), label=name)
for k in range(2):
    ax[k].axvspan(mpar.awin_lo, mpar.awin_hi, color="gray", alpha=0.1)
    ax[k].set_xlabel("assets a")
ax[0].set(title="Error against the exact policy, max over income states", ylabel="|c - exact|")
ax[1].set(title="Euler equation error, max over income states", ylabel="|e|")
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Check.** The affinity deviation is of the order $10^{-16}$, and every method except VFI on the
grid reproduces the exact policy on the window to better than $10^{-4}$. If not, look at the
stopping rules first. The loops for 2.2 (b) and (c) are yours to write, as in problem set 1.

### 5. Part 3: the CRRA household with log normal income and a borrowing limit

*Questions 3.2 and 3.3.* A household with CRRA utility, $u(c) = c^{1-\gamma}/(1-\gamma)$, income
$\ln z' = \rho \ln z + \sigma_\varepsilon \varepsilon'$ with $\varepsilon' \sim N(0, 1)$,
$\rho = 0.9$, $\sigma_\varepsilon = 0.2$, on a Rouwenhorst chain with $N_z = 7$ points, a borrowing
limit $a' \ge 0$, $r = 0.03$, $\beta = 0.96$, $\gamma = 2$. There is no exact policy. CRRA utility
is defined for $c > 0$ only, so every candidate a solver evaluates has to respect $c > 0$, while the
household itself never chooses $c = 0$, since $u_c(c) \to \infty$ there.

**The model as a parameter structure and as functions (gaps 4 to 10).** Fill in the parameters and
the three utility functions. Each gap names the row of the comparison table of Question 3.1 it
belongs to. `income_log`, `bounds_crra`, and `guess_crra` are given: compare `bounds_crra` with
`bounds_cara`, it is the answer to Question 3.1 (d).

In [ ]:
@dataclass(frozen=True)
class CRRAParameters:
    # CRRA utility, income log normal and persistent, borrowing limit a' >= 0
    gamma: float = ...  # GAP 4: relative risk aversion (row: utility)
    r: float = ...  # GAP 5: net interest rate (row: discounting)
    beta: float = ...  # GAP 6: discount factor, beta R < 1 (row: discounting)
    sigma: float = 0.2 / np.sqrt(1 - 0.9 ** 2)     # unconditional sd of ln z, from sigma_eps = 0.2 and rho = 0.9 (row: income)
    rho: float = ...  # GAP 7: persistence of ln z (row: income)

    @property
    def R(self):
        return 1 + self.r


def util_crra(c, par):
    # u(c) = c^(1 - gamma) / (1 - gamma), defined for c > 0 only (row: utility)
    return ...  # GAP 8: the CRRA utility function


def uprime_crra(c, par):
    return ...  # GAP 9: marginal utility u_c(c) (row: utility)


def uprime_inv_crra(m, par):
    return ...  # GAP 10: the inverse of marginal utility, the c with u_c(c) = m (row: utility)


def income_log(par, mpar):
    # income in logs: ln z on a Rouwenhorst chain with sd sigma and persistence rho, returned in levels
    lnz, Pi = rouwenhorst(mpar.nz, par.rho, par.sigma)
    return np.exp(lnz), Pi


def bounds_crra(par, mpar, gri):
    # admissible consumption at every state: the grid bounds on a', and c > 0 (rows: constraints)
    cmin = np.maximum(gri.x - gri.a[-1], 1e-10)              # save the most, a' = a_N, but never consume c <= 0
    cmax = gri.x - gri.a[0]                                  # borrow the most, a' = a_1 = 0, the borrowing limit
    return cmin, cmax


def guess_crra(par, gri):
    # starting policy for the Euler equation methods: consume income plus interest, c = r a + z (slide 12)
    return par.r * gri.a[:, None] + gri.z[None, :]

**The numerical parameters (gap 11).** The asset grid is the model's too: its lower bound is the
borrowing limit, its upper bound has to hold the stationary wealth distribution, and the
spacing should put points where the policy has its kink. The window is where you report the Euler
error, it should stay clear of the region where the borrowing limit binds for every income state.

In [ ]:
mpar3 = ...  # GAP 11: N_a = 200, and amin, amax, spacing, awin_lo, awin_hi for this model (rows: asset grid, accuracy check)
par3 = CRRAParameters()
gri3 = make_grids(par3, mpar3, income_log)
lam3 = stationary(gri3.Pi)
print(f"income grid z: {np.round(gri3.z, 3)}")
print(f"stationary distribution: {np.round(lam3, 4)}, mean of z {lam3 @ gri3.z:.4f}, sd of ln z {np.sqrt(lam3 @ np.log(gri3.z) ** 2):.4f}")
print(f"asset grid: {mpar3.na} points on [{gri3.a[0]}, {gri3.a[-1]}], first spacings {np.round(np.diff(gri3.a[:4]), 4)}, last spacing {gri3.a[-1] - gri3.a[-2]:.3f}")
print(f"window: {window(mpar3, gri3).sum()} of {mpar3.na} grid points, a in [{mpar3.awin_lo}, {mpar3.awin_hi}]")

**The five methods (Question 3.2).** The same calls as in Part 2 with the CRRA functions. There is
no `errors` line, the Euler equation error is the only check. The iteration counts in the table
below are the ones to reproduce, the other columns are yours to fill in (Question 3.2 b).

| method | iterations | time (s) | ms per iteration | Euler error, max | Euler error, mean $\log_{10}$ |
| --- | --- | --- | --- | --- | --- |
| VFI on the grid | 339 | | | | |
| VFI off the grid | 339 | | | | |
| Howard | 11 | | | | |
| Time iteration | 272 | | | | |
| EGM | 272 | | | | |

In [ ]:
results3 = {}
results3["VFI on the grid"] = vfi_on_grid(par3, mpar3, gri3, util_crra, bounds_crra)
results3["VFI off the grid"] = vfi_off_grid(par3, mpar3, gri3, util_crra, bounds_crra)
results3["Howard"] = howard(par3, mpar3, gri3, util_crra, bounds_crra)
results3["Time iteration"] = time_iteration(par3, mpar3, gri3, uprime_crra, bounds_crra, guess_crra)
results3["EGM"] = egm(par3, mpar3, gri3, uprime_crra, uprime_inv_crra, guess_crra)
for name, res in results3.items():
    report(name, res, par3, mpar3)
    res["euler"] = euler_error(res["c"], par3, mpar3, gri3, uprime_crra, uprime_inv_crra)
    res["ee_max"], res["ee_mean"] = euler_stats(res["euler"], mpar3, gri3)
    print(f"  Euler error on the window: max {res['ee_max']:.1e}, mean log10 {res['ee_mean']:.1f}")

In [ ]:
print(f"{'method':18s} {'iterations':>10s} {'time (s)':>9s} {'ms/iter':>8s} {'EE max':>9s} {'EE mean log10':>14s} {'at a_1 (%)':>11s}")
for name, res in results3.items():
    print(f"{name:18s} {res['iters']:10d} {res['time']:9.2f} {1000 * res['time'] / res['iters']:8.2f} "
          f"{res['ee_max']:9.1e} {res['ee_mean']:14.1f} {100 * res['at_lower']:11.1f}")

The consumption policy at the lowest and the highest income state near the borrowing limit, and
the Euler equation error along the grid.

In [ ]:
sel = gri3.a <= 10.0
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, res in results3.items():
    ax[0].plot(gri3.a[sel], res["c"][sel, 0], label=name)
    ax[0].plot(gri3.a[sel], res["c"][sel, -1], ls=":")
    ee = np.nanmax(np.abs(res["euler"]), axis=1)
    ax[1].semilogy(gri3.a, np.maximum(ee, 1e-16), label=name)
ax[0].set(title=f"Consumption at z = {gri3.z[0]:.2f} (solid) and z = {gri3.z[-1]:.2f} (dotted), a <= 10", xlabel="assets a", ylabel="consumption c")
ax[0].legend(fontsize=8)
ax[1].axvspan(mpar3.awin_lo, mpar3.awin_hi, color="gray", alpha=0.1)
ax[1].set(title="Euler equation error, max over income states (shaded: the window)", xlabel="assets a", ylabel="|e|")
plt.tight_layout()
plt.show()

**Check.** All five methods converge with the iteration counts of the table, and the share of
states at $a_1 = 0$ is the same for all of them up to a few tenths of a percent. If a method
reports states at $a_{N_a}$ or does not converge, look at the grid first. The
experiment of 3.3 is yours to write.